In [1]:
import pandas as pd
import wrds

username = "hauvi"
db = wrds.Connection(wrds_username=username)

Loading library list...
Done


# 1. Build the bond universe

In [2]:
filepath = "data/AGG_holdings_clean.csv"
ishare_df = pd.read_csv(filepath, skiprows=9, header=0)
cols_to_drop = ['Location', 'Exchange', 'Currency','Yield to Call (%)', 'Asset Class']
ishare_df = ishare_df.drop(columns=cols_to_drop)
ishare_df.groupby('Sector')['Weight (%)'].sum().sort_values(ascending=False).head(12) 

Sector
Treasury                   46.79
MBS Pass-Through           21.16
Industrial                  4.87
Financial Institutions      4.52
CMBS                        0.96
Sovereign                   0.92
Supranational               0.88
Cash and/or Derivatives     0.85
Agency                      0.32
Local Authority             0.28
ABS                         0.20
Utility                     0.12
Name: Weight (%), dtype: float64

In [ ]:
sectors = ['Treasury', 'Industrial', 'Financial Institutions', 'Utility', 'MBS Pass-Through']
ishare_df = ishlare_df[ishare_df.Sector.isin(sectors)]
ishare_cusips = ishare_df.CUSIP.unique()

# 2. Clean CRSP Treasury Data

In [4]:
library = 'crsp'
start_date = '2024-01-01'
tfz_daily_query = f"""
    select i.tcusip         as cusip,
           d.caldt          as date,
           i.itype          as security_type,
           i.tcouprt        as coupon_rate,
           i.tdatdt         as dated_date,
           i.tmatdt         as maturity_date,
           d.tdbid          as bid_price,
           d.tdask          as ask_price,
           d.tdnomprc       as mid_price,
           d.tdnomprc_flg   as mid_price_flag,
           d.tdaccint       as accrued_interest,
           d.tdyld          as daily_yield,
           d.tdyld * 365    as annual_yield,
           d.tdduratn       as duration_days,
           d.tdduratn / 365 as duration_years,
           d.tdretnua       as daily_return,
           d.tdtotout       as total_outstanding,
           d.tdpubout       as public_outstanding
    from {library}.tfz_dly as d
    join {library}.tfz_iss as i
      on d.kycrspid = i.kycrspid
    where d.caldt >= '{start_date}'
"""

tfz_daily = db.raw_sql(tfz_daily_query, date_cols=['date', 'dated_date', 'maturity_date'])
tfz_daily.security_type = tfz_daily.security_type.astype('int')

In [5]:
def cusip_check_digit(cusip8):
    total = 0
    for i, ch in enumerate(cusip8.upper()):
        if ch.isdigit():
            v = int(ch)
        elif ch.isalpha():
            v = ord(ch) - ord("A") + 10
        elif ch == "*":
            v = 36
        elif ch == "@":
            v = 37
        elif ch == "#":
            v = 38
        else:
            print(f"Invalid CUSIP character: {ch}") 
        if i % 2 == 1:     
            v *= 2
        total += v // 10 + v % 10
    return str((10 - total % 10) % 10)

def to_cusip9(cusip8):
    if cusip8 is None:
        return None
    return cusip8 + cusip_check_digit(cusip8)

In [6]:
tfz_daily["cusip9"] = tfz_daily.cusip.str.strip().str.upper().apply(to_cusip9)
tfz_cusips = tfz_daily.cusip9.unique()

In [7]:
def cusip_in_ishares(df, cusip_col):
    '''
    filter the df to only include rows where the cusip is in ishares df
    '''
    return df[df[cusip_col].isin(ishare_df.CUSIP)]

In [8]:
tfz_daily_filtered = cusip_in_ishares(tfz_daily, 'cusip9')

# 2. Clean TRACE Data

## MBS pass-through from TRACE Enhanced

In [9]:
start_date = '2024-01-01'

def mbs_query(cusips):
    cusip_list = ",".join(f"'{c}'" for c in cusips)
    return f"""
        select cusip_id                                      as cusip,
               trd_exctn_dt                                  as date,
               sum(rptd_pr * entrd_vol_qt) / sum(entrd_vol_qt) as price_volume_wtd,
               min(rptd_pr)                                  as price_low,
               max(rptd_pr)                                  as price_high,
               sum(entrd_vol_qt)                             as volume,
               count(*)                                      as trade_count,
               max(factor)                                   as factor
        from trace_enhanced.trace_spds_mbs_enhanced
        where trd_exctn_dt >= '{start_date}'
          and cusip_id in ({cusip_list})
          and trc_st = 'T'
          and rptd_pr > 0
          and entrd_vol_qt > 0
        group by cusip_id, trd_exctn_dt
    """

mbs_daily = pd.concat(
    [db.raw_sql(mbs_query(ishare_cusips[i:i+1000]), date_cols=['date'])
     for i in range(0, len(ishare_cusips), 1000)],
    ignore_index=True,
)

In [16]:
mbs_daily

,cusip,date,price_volume_wtd,price_low,price_high,volume,trade_count,factor
0,3128MJYM6,2024-01-02,88.543,88.543,88.543,15000.0,3,0.200245
1,3128MJZB9,2024-01-02,87.6256,87.6256,87.6256,36000.0,4,0.0
2,3128MJZM5,2024-01-02,87.6256,87.6256,87.6256,3000.0,3,0.0
3,3132DWA60,2024-01-02,84.305451,83.575394,84.515625,152000.0,18,0.636356
4,3132DWB69,2024-01-02,84.375,84.375,84.375,6000.0,3,0.0
...,...,...,...,...,...,...,...,...
64464,36179RD71,2024-09-25,95.328125,95.328125,95.328125,10238000.0,1,0.0
64465,36179RD71,2024-10-08,94.265625,94.265625,94.265625,27800000.0,1,0.093593
64466,36179RD71,2024-11-19,90.249838,89.963999,91.09375,1980598.0,4,0.09278
64467,36179RD71,2025-02-19,91.078125,91.078125,91.078125,1690000.0,1,0.0


In [ ]:
mbs_daily['mid_price'] = (mbs_daily.price_low + mbs_daily.price_high) / 2
mbs_daily['']

Index(['cusip', 'date', 'security_type', 'coupon_rate', 'dated_date',
       'maturity_date', 'bid_price', 'ask_price', 'mid_price',
       'mid_price_flag', 'accrued_interest', 'daily_yield', 'annual_yield',
       'duration_days', 'duration_years', 'daily_return', 'total_outstanding',
       'public_outstanding', 'cusip9'],
      dtype='object')

#